# Imports

In [1]:
import os
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

import pickle
import time
import sklearn
import gc

from matplotlib import pyplot as plt

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Normalization

In [3]:
def normalize(img):
    return (img - np.mean(img)) / (np.std(img) + 1e-8)
# Cell 4 — Dataset Indexing
def index_isles_dataset(root_dir):

    images_dir = os.path.join(root_dir, "imagesTr")
    labels_dir = os.path.join(root_dir, "labelsTr")

    cases = {}

    for img_path in glob.glob(os.path.join(images_dir, "*.nii.gz")):

        fname = os.path.basename(img_path)
        parts = fname.split("_")

        case_id = "_".join(parts[:2])
        modality_id = parts[2].split(".")[0]

        if case_id not in cases:
            cases[case_id] = {"dwi":None,"adc":None,"label":None}

        if modality_id == "0000":
            cases[case_id]["dwi"] = img_path
        elif modality_id == "0001":
            cases[case_id]["adc"] = img_path

    for lbl_path in glob.glob(os.path.join(labels_dir,"*.nii.gz")):

        case_id = os.path.basename(lbl_path).split(".")[0]

        if case_id in cases:
            cases[case_id]["label"] = lbl_path

    samples = [
        (v["dwi"],v["adc"],v["label"])
        for v in cases.values()
        if v["dwi"] and v["adc"] and v["label"]
    ]

    print("Indexed",len(samples),"cases")

    return samples

# Dataset Class

In [4]:
class ISLESDataset(Dataset):

    def __init__(self, samples, target_size=256):

        self.items = []
        self.target_size = target_size

        for dwi_path, adc_path, mask_path in samples:

            dwi_vol = nib.load(dwi_path).get_fdata()

            for z in range(dwi_vol.shape[2]):
                self.items.append((dwi_path, adc_path, mask_path, z))

    def __len__(self):
        return len(self.items)

    def _resize(self,img):

        img = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)

        img = F.interpolate(
            img,
            size=(self.target_size,self.target_size),
            mode="bilinear",
            align_corners=False
        )

        return img.squeeze(0).squeeze(0)

    def _resize_mask(self,mask):

        mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)

        mask = F.interpolate(
            mask,
            size=(self.target_size,self.target_size),
            mode="nearest"
        )

        return mask.squeeze(0).squeeze(0)

    def __getitem__(self,idx):

        dwi_path, adc_path, mask_path, z = self.items[idx]

        dwi = normalize(nib.load(dwi_path).get_fdata()[:,:,z])
        adc = normalize(nib.load(adc_path).get_fdata()[:,:,z])
        mask = nib.load(mask_path).get_fdata()[:,:,z]

        dwi = self._resize(dwi)
        adc = self._resize(adc)
        mask = self._resize_mask(mask)

        img = torch.stack([dwi,adc],dim=0)
        mask = (mask>0).long()

        return img.float(), mask

# Model Blocks

In [5]:
class DoubleConv(nn.Module):

    def __init__(self,in_ch,out_ch):

        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,out_ch,3,padding=1),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        return self.conv(x)

In [6]:
class AttentionGate(nn.Module):

    def __init__(self,F_g,F_l,F_int):

        super().__init__()

        self.W_g = nn.Conv2d(F_g,F_int,1)
        self.W_x = nn.Conv2d(F_l,F_int,1)
        self.psi = nn.Conv2d(F_int,1,1)

    def forward(self,g,x):

        psi = F.relu(self.W_g(g)+self.W_x(x))
        psi = torch.sigmoid(self.psi(psi))

        return x * psi

In [7]:
class AttentionUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(2,32)
        self.enc2 = DoubleConv(32,64)
        self.enc3 = DoubleConv(64,128)
        self.enc4 = DoubleConv(128,256)

        self.pool = nn.MaxPool2d(2)

        self.dec3 = DoubleConv(256+128,128)
        self.dec2 = DoubleConv(128+64,64)
        self.dec1 = DoubleConv(64+32,32)

        self.att3 = AttentionGate(256,128,64)
        self.att2 = AttentionGate(128,64,32)
        self.att1 = AttentionGate(64,32,16)

        self.out = nn.Conv2d(32,2,1)

    def forward(self,x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        d3 = F.interpolate(e4,scale_factor=2,mode="bilinear",align_corners=False)
        e3 = self.att3(d3,e3)
        d3 = self.dec3(torch.cat([d3,e3],1))

        d2 = F.interpolate(d3,scale_factor=2,mode="bilinear",align_corners=False)
        e2 = self.att2(d2,e2)
        d2 = self.dec2(torch.cat([d2,e2],1))

        d1 = F.interpolate(d2,scale_factor=2,mode="bilinear",align_corners=False)
        e1 = self.att1(d1,e1)
        d1 = self.dec1(torch.cat([d1,e1],1))

        return self.out(d1)

# Dice Metric

In [8]:
def dice_score(pred,target,eps=1e-6):

    pred = pred.view(-1)
    target = target.view(-1)

    inter = (pred*target).sum()

    return (2*inter+eps)/(pred.sum()+target.sum()+eps)

# Dataset Indexing

In [9]:
TRAIN_ROOT = r"/run/media/debasish/WD Green SSD/Attention-U-net-Stroke-analysis/Dataset/ISLES-2022/"
TEST_ROOT  = r"/run/media/debasish/Toshiba HDD/ankit_test/"
train_samples = index_isles_dataset(TRAIN_ROOT)
test_samples  = index_isles_dataset(TEST_ROOT)

print("Train cases:", len(train_samples))
print("Test cases:", len(test_samples))


Indexed 200 cases
Indexed 50 cases
Train cases: 200
Test cases: 50


In [10]:
train_dataset = ISLESDataset(train_samples)
test_dataset  = ISLESDataset(test_samples)

# DataLoaders

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True
)

# Initialize Model

In [12]:
model = AttentionUNet().to(device)

# Loss + Optimizer

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# Training Settings

In [14]:
EPOCHS = 200

checkpoint_dir = "/run/media/debasish/WD Green SSD/Attention-U-net-Stroke-analysis/Scripts/checkpoints_attention_unet/"
os.makedirs(checkpoint_dir, exist_ok=True)

if os.path.exists(os.path.join(checkpoint_dir, "best_dice_info")):
    with open(os.path.join(checkpoint_dir, "best_dice_info"), "rb") as infile:
        best_dice_info = pickle.load(infile)
        best_dice = best_dice_info["best_dice"]
        best_dice_epoch = best_dice_info["best_dice_epoch"]
else:
    best_dice = 0
    best_dice_epoch = 0

print(f"Best dice is {best_dice} @ epoch {best_dice_epoch}")

Best dice is 0.6448061899798618 @ epoch 11


# Training + Test Evaluation Loop

In [ ]:
train_losses = []
test_dices = []

epoch_times = []

for epoch in range(1, EPOCHS + 1):
    epoch_file_path = os.path.join(checkpoint_dir, f"epoch_{epoch}.pth")
    if os.path.exists(epoch_file_path):
        model_data = torch.load(epoch_file_path)
        optimizer.load_state_dict(model_data["optimizer_state"])
        model.load_state_dict(model_data["model_state"])
        train_losses = model_data["train_losses"]
        test_dices = model_data["test_dices"]
        print(f"Skipping epoch {epoch}")
        continue

    start_time = time.perf_counter()
    gc.collect()

    model.train()
    running_loss = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch} Train"):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # -----------------------------
    # VALIDATION SET EVALUATION
    # -----------------------------
    
    model.eval()

    dices = []

    with torch.no_grad():

        for x, y in tqdm(test_loader, desc=f"Epoch {epoch} Test"):

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            preds = torch.argmax(logits, dim=1)

            d = dice_score(
                (preds == 1).float(),
                (y == 1).float()
            ).item()

            dices.append(d)

    mean_dice = np.mean(dices)
    test_dices.append(mean_dice)

    print(f"\nEpoch {epoch}")
    print("Train Loss:", train_loss)
    print("Test Dice:", mean_dice)

    # -----------------------------
    # SAVE CHECKPOINT
    # -----------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_losses": train_losses,
            "test_dices": test_dices
        },
        epoch_file_path
    )

    # -----------------------------
    # SAVE BEST MODEL
    # -----------------------------

    if mean_dice > best_dice:

        best_dice = mean_dice
        best_dice_epoch = epoch

        with open(os.path.join(checkpoint_dir, "best_dice_info"), "wb+") as outfile:
            pickle.dump({
                "best_dice": best_dice,
                "best_dice_epoch": epoch
            }, outfile)

        torch.save(
            model.state_dict(),
            os.path.join(checkpoint_dir, "best_model.pth")
        )

        print("Best model updated!")

    else:
        print(f"{epoch - best_dice_epoch} epochs without improvement")

    # Save plots

    plots = plt.figure()
    plt.subplot(2,1,1)
    plt.plot(train_losses)
    plt.title("Training Loss")
    plt.xlabel("Epoch")
    plt.subplot(2,1,2)
    plt.plot(test_dices)
    plt.title("Test Dice")
    plt.xlabel("Epoch")
    plt.savefig("plots.png")
    plt.close(plots)
    

    end_time = time.perf_counter()
    time_taken_by_epoch = end_time - start_time
    epoch_times.append(time_taken_by_epoch)
    rem_time = (EPOCHS - epoch) * np.mean(np.array(epoch_times))
    print(f"Epoch completed in {time_taken_by_epoch:.2f}s. ETA: {rem_time // 3600}h {(rem_time % 3600) // 60}m")

Skipping epoch 1
Skipping epoch 2
Skipping epoch 3
Skipping epoch 4
Skipping epoch 5
Skipping epoch 6
Skipping epoch 7


/tmp/ipykernel_11788/1779247687.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(epoch_file_path)


Skipping epoch 8
Skipping epoch 9
Skipping epoch 10
Skipping epoch 11


Epoch 12 Train:   0%|          | 0/822 [00:00<?, ?it/s]

Epoch 12 Test:   0%|          | 0/159 [00:00<?, ?it/s]

...


Epoch 26
Train Loss: 0.005347147650733003
Test Dice: 0.671572677146082
1 epochs without improvement
Epoch completed in 259.74s. ETA: 12.0h 37.0m


Epoch 27 Train:   0%|          | 0/822 [00:00<?, ?it/s]

Epoch 27 Test:   0%|          | 0/159 [00:00<?, ?it/s]


Epoch 27
Train Loss: 0.00466450235922188
Test Dice: 0.694291576911066
2 epochs without improvement
Epoch completed in 259.66s. ETA: 12.0h 32.0m


Epoch 28 Train:   0%|          | 0/822 [00:00<?, ?it/s]

# Training Curves

In [ ]:
assert False

import matplotlib.pyplot as plt

train_losses = []
val_dices = []
for epoch_file in glob.glob(os.path.join(checkpoint_dir, "epoch_*.pth")):
    print(epoch_file)
    data = torch.load(epoch_file)
    train_losses.append(data["train_losses"][-1])
    val_dices.append(data["test_dices"][-1])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")

plt.subplot(1,2,2)
plt.plot(test_dices)
plt.title("Validation Dice")
plt.xlabel("Epoch")

plt.show()